# TCI Simulator Course — Weeks 3–7

### Pharmacokinetic Simulation for Anesthesia Practitioners

**Parts II & III — the engine, from one equation to a real solver.**

This notebook is the *learning* half of the course. Each week follows the same
rhythm:

1. **Concept** — the pharmacology, stated as an equation.
2. **By hand** — arithmetic you can check on paper.
3. **The code** — a runnable cell that mirrors the equation line-for-line.
4. **Run & check** — confirm the code agrees with your arithmetic.
5. **Exercises** — with worked solutions hidden under a *Show Solution* toggle,
   so you can attempt them first.

Weeks 8–16 (the reusable model library) live in the companion Word document.
The seam is Week 7's `pkpd_deriv` function — the last thing you *learn* here and
the first thing the library *wraps*.

> **Not a dosing device.** This is a teaching and simulation tool. Parameters
> are for learning the mechanics, not for clinical use.


## Setup

Run this first. It loads the libraries used throughout. Weeks 3–6 need only
`math` and `matplotlib`; Week 7 adds `numpy` and SciPy's `solve_ivp`.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

print("Setup complete.")

Setup complete.


---
# PART II — ONE TANK, THEN THREE (Weeks 3–6)


## Week 3 — The Single Tank and Exponential Decay

**Goal:** understand the one equation behind all PK, and the exact solution for
a single bolus.

**Builds toward:** Euler (Week 4) approximates exactly this, so you can later
trust it.

### Concept

Picture the body as a tank of volume $V$ holding an amount $A$ of drug.
First-order elimination removes drug in proportion to how much is present, with
rate constant $k$ (per minute). All of pharmacokinetics starts as one sentence —
*the rate of change of the amount equals what flows in minus what flows out:*

$$\frac{dA}{dt} = (\text{infusion}) - k\,A$$

Concentration is $C = A/V$. With a single bolus $D$ and no infusion, the amount
decays exponentially:

$$A(t) = D\,e^{-kt}, \qquad C(t) = \frac{D}{V}\,e^{-kt}, \qquad t_{1/2} = \frac{\ln 2}{k} \approx \frac{0.693}{k}$$

### By hand

$D = 100$ mg, $V = 30$ L, $k = 0.10$ /min:

- $C(0) = 100/30 = 3.33$ mg/L
- Elimination rate at $t=0$: $k\,A = 0.10 \times 100 = 10$ mg/min
- $t_{1/2} = 0.693/0.10 = 6.93$ min; check $A(6.93) = 100\,e^{-0.693} = 50$ mg ✓
- $C(6.93) = 50/30 = 1.667$ mg/L — half the start. *Write these down.*


In [2]:
import math  # math.exp (e^x) and math.log (natural log)

D = 100.0   # mg bolus
V = 30.0    # litres
k = 0.10    # per minute

print('C(0) =', round(D / V, 3), 'mg/L')            # 3.333
for t in [0, 1, 6.93, 10, 30]:                       # a few time points
    A = D * math.exp(-k * t)                          # exact amount remaining
    print('t =', t, 'min   C =', round(A / V, 3), 'mg/L')

half_life = math.log(2) / k                           # ln(2)/k
print('half-life =', round(half_life, 2), 'min')     # 6.93

C(0) = 3.333 mg/L
t = 0 min   C = 3.333 mg/L
t = 1 min   C = 3.016 mg/L
t = 6.93 min   C = 1.667 mg/L
t = 10 min   C = 1.226 mg/L
t = 30 min   C = 0.166 mg/L
half-life = 6.93 min


### Run & check

Confirm $t = 6.93 \to C = 1.667$ mg/L and half-life $= 6.93$ min, matching your
paper. The exact curve and your arithmetic agree — the standard you will hold
every approximation to.

*Reference: Abuhelwa AY, Foster DJR, Upton RN. ADVAN-style analytical solutions.
J Pharmacol Toxicol Methods. 2015;73:42-48.*

### Exercise 3.1 — Half-Life to Rate Constant

**Given:** a drug has a half-life of 20 minutes.
**Find:** (a) the rate constant $k$ (/min); (b) $C(30\text{ min})$ if $C(0)=5.0$
mg/L; (c) the time for concentration to fall to 0.5 mg/L.

*Hint: $k = \ln 2 / t_{1/2}$ and $C(t)=C(0)e^{-kt}$. For (c), rearrange to
$t = -\ln(C/C_0)/k$.*

In [ ]:
# Exercise 3.1 — your work here.
# (Try it before revealing the solution below.)

<details>
<summary><b>Show Solution 3.1</b></summary>

**(a)** $k = \ln 2 / 20 = 0.693/20 = $ **0.0347 /min**

**(b)** $C(30) = 5.0\,e^{-0.0347\times30} = 5.0\,e^{-1.040} = 5.0\times0.354 = $ **1.77 mg/L**.
Check: at $t=20$ (one half-life) $C=2.5$; at $t=40$, $C=1.25$. So $C(30)$ sits
between 2.5 and 1.25. ✓

**(c)** $t = -\ln(0.5/5.0)/0.0347 = -\ln(0.1)/0.0347 = 2.303/0.0347 = $ **66.4 min**
$\approx 3.3$ half-lives. Check: after 3 half-lives (60 min) $C=5/8=0.625$; after
4 (80 min) $C=5/16=0.313$. So 66 min is right. ✓

</details>

### Exercise 3.2 — Amount vs Concentration, and What Volumes Mean

**Given:** two patients each receive a 200 mg propofol bolus. Patient A:
$V_1=10$ L, $k=0.08$ /min. Patient B: $V_1=20$ L, $k=0.08$ /min.
**Find:** (a) $C(0)$ for each; (b) time to reach $C=2.0$ µg/mL for each; (c) are
the amounts in the body the same at that time? Explain.

*Insight target: a larger volume of distribution dilutes the initial
concentration but does not change the amount remaining at any given time.*

In [ ]:
# Exercise 3.2 — your work here.

<details>
<summary><b>Show Solution 3.2</b></summary>

**(a)** A: $C(0)=200/10=$ **20 mg/L**. B: $C(0)=200/20=$ **10 mg/L**.

**(b)** $t = -\ln(C_\text{target}/C_0)/k$.
A: $t = -\ln(2/20)/0.08 = \ln(10)/0.08 = 2.303/0.08 = $ **28.8 min**.
B: $t = -\ln(2/10)/0.08 = \ln(5)/0.08 = 1.609/0.08 = $ **20.1 min**.

**(c)** Amount $= C\times V$. At target: A has $2.0\times10=20$ mg; B has
$2.0\times20=40$ mg. **Not the same.** B reaches the same plasma concentration
sooner, but carries twice the body load — the crux of why large-$V_d$ patients
behave differently even at identical plasma targets.

</details>

---
## Week 4 — Euler's Method: Simulating Time as a Loop

**Goal:** march a model forward in tiny time steps — the simulation engine in its
simplest form.

**Builds on:** the single-tank equation (Week 3). **Builds toward:** the same
loop extended to three tanks (Week 5) and an effect site (Week 6).

### Concept

When the input keeps changing (boluses, infusions on/off), we cannot use the
clean exponential. So we step through time: from the current amount, compute the
rate of change, multiply by a tiny step $dt$, add it on, and repeat. That is
Euler's method:

$$A(t + dt) = A(t) + \frac{dA}{dt}\,dt$$

Small $dt$ → accurate but more steps. Large $dt$ → fast but wrong. We learn Euler
because it makes the model transparent; Week 7 replaces it with a solver that
controls its own error.

### By hand

Single tank: $A_0=100$, $k=0.1$, $dt=1$.

| Step / t | A before → dA/dt = −kA → A after |
|---|---|
| 0 / t=0 | 100.00 → −10.00 → 90.00 |
| 1 / t=1 | 90.00 → −9.00 → 81.00 |
| 2 / t=2 | 81.00 → −8.10 → 72.90 |

At $t=2$, Euler gives 72.90; the exact value is $100\,e^{-0.2}=81.87$. Big $dt$ =
big error. Shrink $dt$ for accuracy — the intuition for why Week 7 hands stepping
to a proper solver.

### The code

The `while` loop body **is** Euler exactly: compute the rate of change, update the
amount, advance $t$, then record. Run it, then **change `dt` to `1.0` and rerun**
— watch the Euler curve sag.

In [ ]:
import math
import matplotlib.pyplot as plt

D, V, k = 100.0, 30.0, 0.10
dt = 0.1          # minutes per step  <-- try changing to 1.0
t_end = 30.0

amount = D        # at t=0 the whole bolus is in the tank
t = 0.0
times = [0.0]
concs = [amount / V]

while t < t_end:                          # loop until we reach t_end
    rate_of_change = -k * amount          # dA/dt
    amount = amount + rate_of_change * dt  # Euler update
    t = t + dt
    times.append(t)
    concs.append(amount / V)

exact = [(D / V) * math.exp(-k * tt) for tt in times]   # exact curve

plt.figure(figsize=(7, 4))
plt.plot(times, concs, label='Euler dt=%.2f' % dt)
plt.plot(times, exact, '--', label='Exact')
plt.xlabel('Time (min)'); plt.ylabel('Concentration (mg/L)')
plt.legend(); plt.title('Single tank: Euler vs exact'); plt.show()

### Run & check

With $dt=0.1$ the two curves nearly overlap. Change to $dt=1.0$ and the Euler
curve visibly sags. You have now *seen* discretization error — and the motivation
for Week 7.

### Exercise 4.1 — Euler by Hand: Three Steps and Error Analysis

**Given:** $A=80$ mg at $t=0$, $k=0.05$ /min, $dt=2$ min, no infusion.
**Task:** complete the table for three Euler steps (through $t=6$), compute the
exact value at each time, and the absolute error.

| t | A (Euler) | dA/dt = −kA | A next | A (exact) | Error |
|---|---|---|---|---|---|
| 0 | 80.00 | | | | |
| 2 | | | | | |
| 4 | | | | | |
| 6 | | — | | | |

*Reflection: how would halving $dt$ change the error?*

The cell below builds the table for you once you want to check.

In [ ]:
# Exercise 4.1 — checker. Try the table by hand first, then run.
A, k, dt = 80.0, 0.05, 2.0
print(f"{'t':>3} {'A(Euler)':>10} {'dA/dt':>8} {'A(exact)':>10} {'error':>8}")
t = 0.0
for _ in range(4):
    exact = 80.0 * math.exp(-k * t)
    dAdt = -k * A
    print(f"{t:>3.0f} {A:>10.2f} {dAdt:>8.2f} {exact:>10.2f} {A - exact:>8.2f}")
    A = A + dAdt * dt
    t += dt

<details>
<summary><b>Show Solution 4.1</b></summary>

| t | A (Euler) | dA/dt | A next | A exact | Error |
|---|---|---|---|---|---|
| 0 | 80.00 | −4.00 | 72.00 | 80.000 | 0.00 |
| 2 | 72.00 | −3.60 | 64.80 | 72.38 | −0.38 |
| 4 | 64.80 | −3.24 | 58.32 | 65.50 | −0.70 |
| 6 | 58.32 | — | — | 59.27 | −0.95 |

Exact: $A(t)=80\,e^{-0.05t}$ → $A(2)=72.38,\ A(4)=65.50,\ A(6)=59.27$. Euler:
$80\to72\to64.8\to58.32$. Errors grow with each step.

*Halving $dt$ to 1 min: each step's error is ~4× smaller, so the cumulative 6-min
error falls to roughly 0.25 mg — about 1/4 as large.*

</details>

### Exercise 4.2 — When Does Euler Break? Stability Reasoning

**Thought experiment:** you use $dt=30$ min with $k=0.10$ /min. The Euler update
is $A_\text{next} = A_\text{old} + (-k\,A\,dt)$.

**(a)** Compute the multiplier $(1 - k\,dt)$. What happens to $A$ after one step
if $A=100$? After two?
**(b)** What is the largest $dt$ that keeps the multiplier positive? What goes
wrong when $k\,dt > 2$?

In [ ]:
# Exercise 4.2 — your work here.

<details>
<summary><b>Show Solution 4.2</b></summary>

**(a)** Multiplier $= 1 - 0.10\times30 = -2.0$. Step 1: $A = 100\times(-2.0)=-200$
mg. Step 2: $-200\times(-2.0)=+400$ mg. Oscillates wildly — physically
meaningless.

**(b)** The multiplier stays positive when $k\,dt < 1$, i.e. $dt < 10$ min. It
stays $\le 1$ (no growth) when $k\,dt \le 2$, i.e. $dt \le 20$ min. For $k\,dt>2$
the solution oscillates and grows — "drug appearing from nowhere." Real TCI pumps
use $dt \approx 10$ s ($1/6$ min), far below the stability limit.

</details>

---
## Week 5 — Three Compartments by Euler (the Marsh Propofol Model)

**Goal:** simulate a real anesthetic with a central + two peripheral tanks.

**Builds on:** the Euler loop (Week 4). **Builds toward:** the effect site
(Week 6) and the `solve_ivp` engine (Week 7) both extend this structure.

### Concept

Intravenous anesthetics need three tanks: central ($V_1$, blood + vessel-rich
organs), fast peripheral ($V_2$), slow peripheral ($V_3$). Drug shuttles between
$V_1$ and the peripherals; elimination happens only from $V_1$.

| Constant | Role |
|---|---|
| $k_{10}$ | Elimination from central |
| $k_{12}/k_{21}$ | Central ↔ fast peripheral |
| $k_{13}/k_{31}$ | Central ↔ slow peripheral |

$$\frac{dA_1}{dt} = -(k_{10}+k_{12}+k_{13})A_1 + k_{21}A_2 + k_{31}A_3 + \text{infusion}$$
$$\frac{dA_2}{dt} = k_{12}A_1 - k_{21}A_2 \qquad \frac{dA_3}{dt} = k_{13}A_1 - k_{31}A_3$$

**Marsh parameters (weight-based):** $V_1 = 0.228\ \text{L/kg}\times\text{weight}$;
$k_{10}=0.119$, $k_{12}=0.112$, $k_{13}=0.0419$, $k_{21}=0.055$, $k_{31}=0.0033$ /min.

### By hand

70 kg → $V_1 = 0.228\times70 = 15.96$ L. Give a 100 mg bolus into $A_1$ (others
empty). One Euler step, $dt=0.1$:

- $dA_1/dt = -(0.119+0.112+0.0419)\times100 = -27.29$ mg/min; $A_1\to97.27$; $C_p=6.09$ mg/L
- $dA_2/dt = 0.112\times100 = 11.2$; $A_2\to1.12$
- $dA_3/dt = 0.0419\times100 = 4.19$; $A_3\to0.419$

### The code — the critical ordering

> **Compute ALL derivatives using the *old* values, then update.** Updating
> $A_1$ before computing $dA_2$ would corrupt the step.

In [ ]:
import matplotlib.pyplot as plt

# Marsh (70 kg): weight-based V1, published rate constants
weight = 70.0
V1 = 0.228 * weight             # 15.96 L
k10, k12, k13 = 0.119, 0.112, 0.0419
k21, k31 = 0.055, 0.0033

A1, A2, A3 = 150.0, 0.0, 0.0    # 150 mg bolus into the central compartment
dt, t_end = 0.1, 30.0
infusion = 0.0

t = 0.0
times = [0.0]; cp = [A1 / V1]
while t < t_end:
    dA1 = -(k10 + k12 + k13) * A1 + k21 * A2 + k31 * A3 + infusion  # ALL derivatives
    dA2 = k12 * A1 - k21 * A2                                        # with OLD values
    dA3 = k13 * A1 - k31 * A3
    A1 = A1 + dA1 * dt                                               # THEN update
    A2 = A2 + dA2 * dt
    A3 = A3 + dA3 * dt
    t += dt
    times.append(t); cp.append(A1 / V1)

plt.figure(figsize=(7, 4))
plt.plot(times, cp)
plt.xlabel('Time (min)'); plt.ylabel('Cp (mg/L)')
plt.title('Marsh propofol — central compartment (Euler)'); plt.show()
print('Cp(0) =', round(150 / V1, 2), 'mg/L  (fast early fall = redistribution)')

### Exercise 5.1 — Two Euler Steps for the Marsh Model

**Patient:** 70 kg. Marsh: $V_1=15.96$ L, $k_{10}=0.119$, $k_{12}=0.112$,
$k_{13}=0.0419$, $k_{21}=0.055$, $k_{31}=0.0033$ /min.
**Dose:** 150 mg bolus at $t=0$ ($A_1=150$, $A_2=A_3=0$). $dt=0.5$ min.
**Task:** two Euler steps. Record $A_1,A_2,A_3$ and $C_p=A_1/V_1$ at $t=0,0.5,1.0$.
Compute all three derivatives before updating any.

In [ ]:
# Exercise 5.1 — your work here (or adapt the Week 5 loop with dt=0.5).

<details>
<summary><b>Show Solution 5.1</b></summary>

**t=0:** $A_1=150,\ A_2=0,\ A_3=0,\ C_p=150/15.96=9.40$ µg/mL.

**Step 1 (→ t=0.5):** $dA_1=-0.2729\times150=-40.93$; $dA_2=0.112\times150=16.80$;
$dA_3=0.0419\times150=6.285$.
→ **$A_1=129.53,\ A_2=8.40,\ A_3=3.14$; $C_p=8.12$ µg/mL.**

**Step 2 (→ t=1.0):** $dA_1=-0.2729\times129.53 + 0.055\times8.40 + 0.0033\times3.14
= -35.35+0.462+0.010 = -34.88$.
→ **$A_1=112.09$; $C_p=7.02$ µg/mL** (falls ~2.4 µg/mL in the first minute —
redistribution at work).

</details>

### Exercise 5.2 — Conservation of Mass Check

Using your results from 5.1: at $t=0,0.5,1.0$, compute $A_1+A_2+A_3$. Are they
equal to 150 mg? *Should* they be?

*Insight: redistribution moves drug between compartments but does not eliminate
it. Only $k_{10}$ is the drain.*

In [ ]:
# Exercise 5.2 — your work here.

<details>
<summary><b>Show Solution 5.2</b></summary>

$t=0$: 150. $t=0.5$: $129.53+8.40+3.14=141.07$. $t=1.0$: ≈ 143.1. **Not equal —
and should not be.**

Elimination in step 1: $k_{10}\,A_1\,dt = 0.119\times150\times0.5 = 8.93$ mg.
Check: $150-141.07=8.93$ ✓. If $A_1+A_2+A_3$ stayed at 150, there would be no
elimination — your code would be wrong.

</details>

---
## Week 6 — The Effect Site and $k_{e0}$

**Goal:** model the brain's lag behind plasma — the difference between "drug in
blood" and "drug working."

**Builds toward:** depth-of-anesthesia (Week 14) and wake-up (Week 16) are all
about $C_e$, not $C_p$.

### Concept

Plasma is not where the drug acts. The effect site (brain) is a tiny extra
compartment whose concentration $C_e$ chases $C_p$:

$$\frac{dC_e}{dt} = k_{e0}\,(C_p - C_e)$$

$k_{e0}$ (per minute) sets the chase speed. After a bolus, $C_e$ peaks *after*
$C_p$ (time-to-peak-effect) — the clinical reason to wait 1–2 minutes after a
propofol bolus before laryngoscopy.

### By hand

At an instant where $C_p=6.0$, $C_e=2.0$, $k_{e0}=0.26$:
$$dC_e/dt = 0.26\times(6.0-2.0) = 1.04\ \text{µg/mL/min}$$
Over $dt=0.1$: $C_e \to 2.0 + 1.04\times0.1 = 2.104$. When $C_p=C_e$, $dC_e/dt=0$
(equilibrium).

### The code — adding the effect site to Week 5's loop

Plot both: $C_e$ rises more slowly, peaks later and lower than $C_p$, and crosses
$C_p$ exactly at peak effect.

In [ ]:
import matplotlib.pyplot as plt

weight = 70.0
V1 = 0.228 * weight
k10, k12, k13 = 0.119, 0.112, 0.0419
k21, k31 = 0.055, 0.0033
ke0 = 0.26

A1, A2, A3 = 150.0, 0.0, 0.0
Ce = 0.0
dt, t_end = 0.1, 30.0

t = 0.0
times = [0.0]; cp = [A1 / V1]; ce_list = [Ce]
while t < t_end:
    dA1 = -(k10 + k12 + k13) * A1 + k21 * A2 + k31 * A3
    dA2 = k12 * A1 - k21 * A2
    dA3 = k13 * A1 - k31 * A3
    A1 += dA1 * dt; A2 += dA2 * dt; A3 += dA3 * dt
    Cp_now = A1 / V1
    dCe = ke0 * (Cp_now - Ce)     # the chase equation
    Ce = Ce + dCe * dt            # Euler step for the effect site
    t += dt
    times.append(t); cp.append(Cp_now); ce_list.append(Ce)

plt.figure(figsize=(7, 4))
plt.plot(times, cp, label='Cp (plasma)')
plt.plot(times, ce_list, label='Ce (effect site)')
plt.xlabel('Time (min)'); plt.ylabel('Concentration (mg/L)')
plt.legend(); plt.title('Ce lags Cp and crosses it at peak effect'); plt.show()

### Exercise 6.1 — Tracing the Ce–Cp Gap

Using the results from Ex 5.1: at $t=0$, $C_p=9.40$, $C_e=0$. At $t=0.5$,
$C_p=8.12$. At $t=1.0$, $C_p=7.02$.

**Scenario A ($k_{e0}=0.26$):** compute $C_e$ at $t=0.5$ and $t=1.0$. What is
$(C_p-C_e)$ at each time?
**Scenario B ($k_{e0}=0.80$):** repeat. How different is $C_e$ at $t=1.0$? What
does this mean clinically?

In [ ]:
# Exercise 6.1 — your work here.

<details>
<summary><b>Show Solution 6.1</b></summary>

**Scenario A ($k_{e0}=0.26$):**
$C_e(0.5) = 0 + 0.26\times9.40\times0.5 = 1.22$ µg/mL; $C_p-C_e = 8.12-1.22 = 6.90$.
$C_e(1.0) = 1.22 + 0.26\times(8.12-1.22)\times0.5 = 1.22 + 0.897 = 2.12$ µg/mL.

**Scenario B ($k_{e0}=0.80$):**
$C_e(0.5) = 0.80\times9.40\times0.5 = 3.76$ µg/mL.
$C_e(1.0) = 3.76 + 0.80\times(8.12-3.76)\times0.5 = 5.50$ µg/mL.

$C_e$ is 2.12 vs 5.50 µg/mL at $t=1$ — the fast-$k_{e0}$ model has far more
effect-site drug at 1 minute. Clinically: $k_{e0}=0.26$ (Marsh) demands waiting
longer before laryngoscopy than $k_{e0}=0.456$ (Schnider).

</details>

### Exercise 6.2 — Time to Peak Effect: Qualitative Reasoning

After a single bolus, $C_p$ peaks at $t=0$ then falls. $C_e$ starts at 0 and
chases $C_p$.

**(a)** What is $dC_e/dt$ when $C_p>C_e$? When $C_p<C_e$? When $C_p=C_e$?
**(b)** At the moment $C_e$ is at its peak, what must $dC_e/dt$ equal? Therefore
what must $C_p$ equal at that instant?
**(c)** A colleague says "just wait 30 seconds." Given $k_{e0}=0.26$ /min, is 30
seconds near peak effect?

In [ ]:
# Exercise 6.2 — your work here.

<details>
<summary><b>Show Solution 6.2</b></summary>

**(a)** $dC_e/dt = k_{e0}(C_p-C_e)$. $C_p>C_e$ → positive (rising); $C_p<C_e$ →
negative (falling); $C_p=C_e$ → zero (momentarily flat).

**(b)** At the $C_e$ peak, $dC_e/dt=0$, so $C_p=C_e$ at that instant — the two
curves cross exactly at peak effect.

**(c)** $1/k_{e0}\approx3.8$ min time constant; time-to-peak-effect ≈ 2–3 min. 30
seconds is far too short — the effect site has barely begun to equilibrate. Wait
at least 1–2 min after a slow bolus, 3 min after rapid induction.

</details>

---
# PART III — THE WORKING ENGINE (Week 7)


## Week 7 — From Euler to `solve_ivp`: One Derivative Function

**Goal:** replace the hand-written Euler loop with a robust solver, while keeping
the code shaped exactly like the PK equations.

**Builds toward:** every model from Week 8 on rides this engine.

### Concept

Euler accumulates error and forces you to choose $dt$. The fix reads more simply:
write the model as one function returning rates of change, and hand it to SciPy's
`solve_ivp`, which steps with automatic error control.

Bundle the three amounts and the effect-site concentration into one state list
$y = [A_1, A_2, A_3, C_e]$. The function below **is** the entire model — PK and
effect site together — reading line-for-line like the pharmacology.

In [ ]:
from scipy.integrate import solve_ivp

def pkpd_deriv(t, y, infusion_func, V1, k10, k12, k13, k21, k31, ke0):
    A1, A2, A3, Ce = y
    Cp = A1 / V1
    rate = infusion_func(t)
    dA1 = rate - (k10 + k12 + k13) * A1 + k21 * A2 + k31 * A3
    dA2 = k12 * A1 - k21 * A2
    dA3 = k13 * A1 - k31 * A3
    dCe = ke0 * (Cp - Ce)
    return [dA1, dA2, dA3, dCe]

### Calling `solve_ivp`

- `method='LSODA'` — auto-switches between gentle and stiff problems; a safe
  default for compartment models.
- `max_step=0.5` — caps the step at ½ min so the solver cannot step over a brief
  infusion-rate change.

A full runnable example: a constant 10 mg/min infusion for 30 min, then off.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Marsh params (70 kg)
V1 = 0.228 * 70
k10, k12, k13, k21, k31 = 0.119, 0.112, 0.0419, 0.055, 0.0033
ke0 = 0.26

def infusion_func(t):
    return 10.0 if 0.0 <= t < 30.0 else 0.0   # mg/min for 30 min

params = (infusion_func, V1, k10, k12, k13, k21, k31, ke0)
y0 = [0.0, 0.0, 0.0, 0.0]    # [A1, A2, A3, Ce] start empty
t_end = 60.0
times = np.linspace(0, t_end, 601)

sol = solve_ivp(pkpd_deriv, (0.0, t_end), y0, args=params,
                method='LSODA', max_step=0.5, t_eval=times)

Cp = sol.y[0] / V1    # row 0 is A1
Ce = sol.y[3]         # row 3 is Ce

plt.figure(figsize=(7, 4))
plt.plot(sol.t, Cp, label='Cp'); plt.plot(sol.t, Ce, label='Ce')
plt.xlabel('Time (min)'); plt.ylabel('Concentration (mg/L)')
plt.legend(); plt.title('solve_ivp: 10 mg/min infusion for 30 min'); plt.show()

### The one thing to get right: boluses are jumps

A bolus is an instantaneous jump in $A_1$. Integrate **piecewise**: split the
timeline at bolus times, integrate each interval, add the bolus to the state at
its event time, and carry the ending state into the next interval.

The example below gives a 100 mg bolus at $t=0$ and a 50 mg bolus at $t=20$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def zero_infusion(t):
    return 0.0

params = (zero_infusion, V1, k10, k12, k13, k21, k31, ke0)

boluses = [(0.0, 100.0), (20.0, 50.0)]   # (time_min, dose_mg)
t_end = 60.0

bolus_times = sorted(bt for bt, _ in boluses)
bounds = sorted(set([0.0] + [bt for bt in bolus_times if 0.0 < bt < t_end] + [t_end]))

y = [0.0, 0.0, 0.0, 0.0]
all_t, all_y = [], []
for i in range(len(bounds) - 1):
    ta, tb = bounds[i], bounds[i + 1]
    for (bt, dose) in boluses:
        if abs(bt - ta) < 1e-9:
            y[0] += dose                 # add bolus to central compartment
    seg_t = np.linspace(ta, tb, int((tb - ta) * 10) + 1)
    sol = solve_ivp(pkpd_deriv, (ta, tb), y, args=params,
                    method='LSODA', max_step=0.5, t_eval=seg_t)
    all_t.append(sol.t); all_y.append(sol.y)
    y = list(sol.y[:, -1])               # carry final state forward

t = np.concatenate(all_t)
Y = np.concatenate(all_y, axis=1)
Cp = Y[0] / V1; Ce = Y[3]

plt.figure(figsize=(7, 4))
plt.plot(t, Cp, label='Cp'); plt.plot(t, Ce, label='Ce')
plt.xlabel('Time (min)'); plt.ylabel('Concentration (mg/L)')
plt.legend(); plt.title('Piecewise: 100 mg bolus at t=0, 50 mg at t=20'); plt.show()

### Exercise 7.1 — Reading the Derivative Function as Pharmacology

For each change, predict whether $C_p$ will (i) start higher, (ii) fall faster, or
(iii) be delayed — and explain using the equation, not just intuition.

**(a)** Halve $V_1$ (same dose). **(b)** Double $k_{10}$. **(c)** Set
$k_{12}=k_{13}=k_{21}=k_{31}=0$ (single-compartment drug). **(d)** Double
$k_{e0}$.

In [ ]:
# Exercise 7.1 — try predicting, then test by editing the params above and re-running.

<details>
<summary><b>Show Solution 7.1</b></summary>

**(a)** $C_p=A_1/V_1$ → halving $V_1$ doubles $C_p(0)$ for the same dose. The
amount trajectory is unchanged; only the concentration is scaled up 2×.

**(b)** Doubling $k_{10}$ doubles the $-k_{10}A_1$ elimination term in $dA_1$. Drug
leaves faster; $C_p$ falls faster. $C_p(0)$ is unchanged (set by dose/$V_1$).

**(c)** $dA_1 = -k_{10}A_1 + \text{infusion}$ — a pure single exponential. Falls
faster initially (no peripheral buffering); the terminal decline is simpler.

**(d)** $dC_e = k_{e0}(C_p-C_e)$. Doubling $k_{e0}$ makes $C_e$ chase $C_p$ twice
as fast — shorter time-to-peak-effect, faster onset and offset. $C_p$ itself is
unaffected.

</details>

### Exercise 7.2 — Why Boluses Are Jumps, and Why That Matters

**Scenario:** 100 mg bolus at $t=0$, then 50 mg bolus at $t=20$ min.

**(a)** Why can't you model a bolus as a very high infusion rate for a very short
$dt$ (e.g. 100 mg / 0.001 min for 0.001 min)? What goes wrong numerically?
**(b)** Describe the piecewise strategy (integrate → jump → integrate) for this
scenario. How many intervals? What is added to $A_1$ at $t=20$?

In [ ]:
# Exercise 7.2 — your work here.

<details>
<summary><b>Show Solution 7.2</b></summary>

**(a)** LSODA chooses its own step size — it may skip right over the tiny window,
delivering nothing. Even with `max_step` forced to $dt$, the enormous rate
(100,000 mg/min) makes $dA_1/dt$ huge, forcing tiny steps everywhere and
accumulating floating-point error. Jumps are exact; high-rate infusions for tiny
windows are not.

**(b)** Two intervals: $[0,20]$ and $[20, t_\text{end}]$. At the start of interval
1 add 100 mg to $A_1$; integrate $[0,20]$. At $t=20$, take the ending state, add
50 mg to $A_1$, integrate $[20, t_\text{end}]$. Two smooth curves joined at $t=20$
with a discontinuous jump in $A_1$ — exactly what a bolus is.

</details>

---
## The handoff to Weeks 8–16

`pkpd_deriv` above is the seam. Everything you built by hand — three compartments,
the effect site, piecewise boluses — now lives in one function with automatic
error control.

In the companion Word document (**Weeks 8–16**), this same function gets wrapped:

- **Week 8** — a `Patient` object owns body-size numbers; a `ThreeCompartmentModel`
  base class wraps `pkpd_deriv` and the piecewise integrator.
- **Weeks 9–11** — real drug models (Marsh, Schnider, Minto, Fentanyl, Hannivoort,
  Eleveld, ketamine, methadone) as subclasses.
- **Weeks 12–13** — dosing engine, unit converters, ketofol, the Roberts regimen.
- **Weeks 14–16** — the Hill/BIS model, the Greco response surface, and the
  wake-up predictor.

You now trust the engine because you built it and checked it against arithmetic.
That is the whole point of Part II & III.